In [ ]:
from fast_rsm.distance_calculator import easyloader, get_line_profiles, identify_peaks, fit_linear_models, calc_dist
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
import os
from itertools import groupby
from operator import itemgetter

In [ ]:

loader = easyloader(nexusfile=nexusfile, datapath=datapath)
selection = np.arange(imstart,imend, 1)
line_profiles = get_line_profiles(loader, selection, central_pixel, move_axis, clip_values)

In [ ]:

tracked_peaks, finished_peaks = identify_peaks(line_profiles)
fullpeaks = [
    peak for peak in finished_peaks + tracked_peaks if len(peak.pos_list) > 1
]

In [ ]:
def plot_peak_details(peak):
    fig,axs=plt.subplots(3,1)
    axlist=axs.ravel()
    axlist[0].plot(peak.pos_list,marker='o')
    axlist[0].set_ylabel('pixel positions')
    axlist[1].plot(peak.imlist,marker='o')
    axlist[1].set_ylabel('image list')
    example_im_ind=peak.imlist[int(np.ceil(len(peak.imlist)/2))]+imstart
    example_im=loader.get_image(example_im_ind)
    axlist[2].imshow(example_im,vmax=example_im.mean()*1.5) 
    axlist[2].set_title(f'image: {example_im_ind} \nimage range {peak.imlist[0]}-{peak.imlist[-1]}')   
    plt.tight_layout()

In [ ]:
for peak in fullpeaks:
    plot_peak_details(peak)

In [ ]:
print(f"found {len(fullpeaks)} good tracked signals")
slopes, slopes_meanerr = fit_linear_models(fullpeaks)

avslope = slopes.mean()
print(f"average_slope = {avslope} +/- {slopes_meanerr}")

pixsize = loader.diff.data_file.pixel_size
selected_angles = loader.diff.data_file.default_axis[selection]
selected_angles_end = selected_angles[1:]
ang_shift = np.array(
    [
        end_angle - selected_angles[i]
        for i, end_angle in enumerate(selected_angles_end)
        ]
    )
print(f"selected angles: {selected_angles[0]}-{selected_angles[-1]}")
print(f"average angle shift = {ang_shift.mean()}")
d1 = calc_dist(ang_shift.mean(), avslope * pixsize)
d2 = calc_dist(ang_shift.mean(), (avslope + (slopes_meanerr)) * pixsize)

print(f"distance (m) = {d1} +/- {d2 - d1}")